Imports & Device Setup

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

from sequence_dataset import VideoSequenceDataset
from dataset_and_transforms import val_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


Load Video-Sequence Dataset

In [2]:
SEQ_LEN = 10       # frames per video
BATCH_SIZE = 2     # video sequences are heavy

train_dataset = VideoSequenceDataset(
    metadata_csv="dataset/processed/metadata_train.csv",
    transform=val_transforms,
    seq_len=SEQ_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Total video samples:", len(train_dataset))


Total video samples: 75


CNN + LSTM Model Definition

In [5]:
class CNN_LSTM(nn.Module):
    def __init__(self, hidden_dim=256, num_classes=2):
        super(CNN_LSTM, self).__init__()

        # CNN Backbone (ResNet50)
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 2048

        # LSTM for temporal learning
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        batch, seq_len, c, h, w = x.shape
        cnn_features = []

        # Extract features frame by frame
        for t in range(seq_len):
            f = self.cnn(x[:, t])
            f = f.view(batch, -1)
            cnn_features.append(f)

        # Shape: (batch, seq_len, feature_dim)
        cnn_features = torch.stack(cnn_features, dim=1)

        # LSTM
        lstm_out, _ = self.lstm(cnn_features)

        # Use last time step
        out = self.fc(lstm_out[:, -1])
        return out


Initialize Model, Loss, Optimizer

In [6]:
model = CNN_LSTM().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(model)


CNN_LSTM(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64, 256

Training Loop

In [7]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    correct = 0
    total = 0
    running_loss = 0

    for sequences, labels in train_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(sequences)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_loss:.4f} | Accuracy: {acc:.4f}")


Epoch [1/5] | Loss: 25.1493 | Accuracy: 0.6267
Epoch [2/5] | Loss: 12.8292 | Accuracy: 0.9067
Epoch [3/5] | Loss: 5.6809 | Accuracy: 0.9733
Epoch [4/5] | Loss: 2.3640 | Accuracy: 0.9733
Epoch [5/5] | Loss: 0.5325 | Accuracy: 1.0000


Save Trained Model

In [8]:
torch.save(model.state_dict(), "model2_cnn_lstm.pth")
print("Model saved as model2_cnn_lstm.pth")


Model saved as model2_cnn_lstm.pth
